In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

url = "https://www.flipkart.com/google-pixel-7-pro-obsidian-128-gb/product-reviews/itmb74dc5c3b3eb5?page=1"

In [ ]:
reviews = []

for page in range(1, 101):   
    
    print(f"Page {page}")
    
    url = f"https://www.flipkart.com/google-pixel-7-pro-obsidian-128-gb/product-reviews/itmb74dc5c3b3eb5?page={page}"

    # https://www.amazon.co.uk/product-reviews/B07R49KYCF/ref=cm_cr_getr_d_paging_btm?ie=UTF8&filterByStar=all_stars&reviewerType=all_reviews&pageNumber={page}&nextPageToken=MjAyNi0wNC0xNVQxMTowODowMC4wNjcyODI3ODBaADEw#reviews-filter-bar
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")
    
    elems = soup.find_all("div", {"dir": "auto"})
    
    for e in elems:
        text = e.get_text(strip=True)
        
        if len(text) > 30:
            reviews.append(text)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
import time
import pandas as pd
import os

# =========================
# CONFIG
# =========================
ASIN = "B07R49KYCF"
PAGES = 15
PROFILE_DIR = r"C:\selenium\edge_profile"  # saves login session

# =========================
# EDGE SETUP (persistent login)
# =========================
options = Options()
options.use_chromium = True

# IMPORTANT: keeps you logged in
options.add_argument(f"user-data-dir={PROFILE_DIR}")

# reduce bot detection
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Edge(options=options)

reviews = []

# =========================
# STEP 1: OPEN AMAZON ONCE (LOGIN STEP)
# =========================
driver.get("https://www.amazon.co.uk/")
time.sleep(5)

input("👉 If not logged in yet, log in now in the browser, then press ENTER...")

# =========================
# STEP 2: SCRAPE REVIEWS
# =========================
for page in range(1, PAGES + 1):
    print(f"Scraping page {page}")

    url = f"https://www.amazon.co.uk/product-reviews/{ASIN}/?pageNumber={page}"
    driver.get(url)
    time.sleep(4)

    # check for CAPTCHA/login redirect
    if "captcha" in driver.page_source.lower():
        input("⚠️ CAPTCHA detected. Solve it, then press ENTER...")

    elems = driver.find_elements(By.XPATH, "//span[@data-hook='review-body']")

    for e in elems:
        text = e.text.strip()
        if text:
            reviews.append(text)

    time.sleep(2)

# =========================
# STEP 3: SAVE CSV
# =========================
driver.quit()

df = pd.DataFrame(reviews, columns=["Review"])
df.to_csv("data/amazon_reviews.csv", index=False)

print(f"✅ Done! Saved {len(reviews)} reviews to amazon_reviews.csv")
    

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6
Scraping page 7
Scraping page 8
Scraping page 9
Scraping page 10
Scraping page 11
Scraping page 12
Scraping page 13
Scraping page 14
Scraping page 15
✅ Done! Saved 150 reviews to amazon_reviews.csv
